# Prise en charge des noms propres (OF3C vers DMF)

Ce notebook sert à :

1. choisir un fichier TSV du corpus OF3C ;
2. extraire les nom propres (par défaut POS = `NOMpro`) et leur fréquence ;
3. charger la liste manuelle `documentation/nomsPropres.ods` ;
4. charger le DMF corrigé `DMF_DICT_2024-05-29/DICT_2024-05-29-corr.xml` ; 
5. produire :
   - un tableau d’alignement (présent DMF/absent/mapping manuel) ;
   - un export TEI minimal pour les noms propres à modéliser/ajouter.

> Remarque : le DMF dans ce dépôt n’est pas du TEI ; on l’exploite ici comme source d’inventaire de lemmes (balise `<LEM>`).

> Attention : ce notebook s’appuie sur des versions antérieures de plusieurs fichiers, en particulier du dictionnaire utilisé pour l’alignement ; la version la plus récente est `DICT.corrected.xml`.

In [ ]:
from pathlib import Path
import pandas as pd
import re
import unicodedata
from datetime import datetime
import json

# Détection de la racine du dépôt

def find_repo_root(start=None):
    if start is None:
        start = Path.cwd()
    for parent in [start] + list(start.parents):
        if (parent / 'OF3C-main').exists() and (parent / 'documentation').exists():
            return parent
    raise RuntimeError('Racine du dépôt introuvable (attendu: OF3C-main/ et documentation/)')

repo_root = find_repo_root()
repo_root

## Choisir un fichier TSV OF3C

In [ ]:
import glob

# Liste tous les TSV disponibles dans OF3C-main/tsv
paths = sorted([Path(p) for p in glob.glob(str(repo_root/'OF3C-main'/'tsv'/'**'/'*.tsv'), recursive=True)])

# Si le dépôt ne contient pas tous les TSV, on propose au moins l’échantillon de documentation/
if not paths:
    sample = repo_root/'documentation'/'nca-sample-naomicorr-corrige.tsv'
    paths = [sample] if sample.exists() else []

print(f"TSV trouvés: {len(paths)}")
for i,p in enumerate(paths[:40]):
    print(f"[{i}] {p.relative_to(repo_root)}")
if len(paths)>40:
    print('…')

# Choix
TSV_INDEX = 0  # <-- changer l’index ici
TSV_PATH = paths[TSV_INDEX]
TSV_PATH

## Charger le TSV et extraire les noms propres

In [ ]:
df = pd.read_csv(TSV_PATH, sep='	')
assert {'form','lemma','POS'}.issubset(df.columns), f"Colonnes attendues manquantes. Colonnes: {list(df.columns)}"

# Quels POS compter comme noms propres ?
PROPN_POS = {'NOMpro'}  # ajoute d'autres étiquettes si besoin

propn = df[df['POS'].isin(PROPN_POS)].copy()

# Nettoyage minimal des lemmes (garde la casse pour l’affichage, mais crée une clé normalisée pour le matching)

def norm_key(s: str) -> str:
    s = '' if pd.isna(s) else str(s).strip()
    # normalisation unicode (accents) + casefold
    s = unicodedata.normalize('NFKC', s)
    return s.casefold()

propn['lemma_key'] = propn['lemma'].map(norm_key)

propn_counts = (
    propn.groupby(['lemma','lemma_key'])
         .size()
         .reset_index(name='count')
         .sort_values('count', ascending=False)
)

print(f"Tokens NOMpro: {len(propn)}")
print(f"Types de lemma NOMpro: {len(propn_counts)}")
propn_counts.head(20)

## Charger la liste manuelle `nomsPropres.ods` (mapping base vers forme DMF)

Le fichier contient généralement :
- `forme retenue base` : la forme canonique côté corpus/référentiel ;
- `forme DMF` : la forme/lemmatisation DMF si elle est connue (sinon vide).

> J'observe des irregularités dans le corpus, notamment pour les noms en latin (quelle forme de référence ? latin, français ?)

In [ ]:
ODS_PATH = repo_root/'documentation'/'nomsPropres.ods'
assert ODS_PATH.exists(), f"ODS introuvable: {ODS_PATH}"

# Dépendance : odfpy est nécessaire pour lire .ods via pandas (engine='odf')
try:
    import odf  # noqa
except Exception:
    # si absent, on l'installe (cellule safe)
    import sys
    !{sys.executable} -m pip install -q odfpy

np_ods = pd.read_excel(ODS_PATH, engine='odf')
np_ods.columns = [c.strip() for c in np_ods.columns]

# attendues
base_col = [c for c in np_ods.columns if 'forme retenue' in c.lower() or 'forme' in c.lower()][0]
dmf_col_candidates = [c for c in np_ods.columns if 'dmf' in c.lower()]
dmf_col = dmf_col_candidates[0] if dmf_col_candidates else None

np_ods = np_ods.rename(columns={base_col:'base_form'})
if dmf_col:
    np_ods = np_ods.rename(columns={dmf_col:'dmf_form'})
else:
    np_ods['dmf_form'] = pd.NA

np_ods['base_key'] = np_ods['base_form'].map(norm_key)
np_ods['dmf_key'] = np_ods['dmf_form'].map(norm_key)

np_ods.head(60)

## Charger le DMF corrigé et extraire l’inventaire des lemmes

Le fichier `DICT_2024-05-29-corr.xml` n’a pas d’en-tête d’encodage ; on le lit en *latin-1* pour éviter les erreurs de parsing.

On extrait ici les valeurs de `<LEM>…</LEM>` pour disposer d’un ensemble de lemmes DMF (c’est volontairement simple)

In [ ]:
DMF_XML = repo_root/'DMF_DICT_2024-05-29'/'DICT_2024-05-29-corr.xml'
assert DMF_XML.exists(), f"DMF XML introuvable: {DMF_XML}"

raw = DMF_XML.read_bytes().decode('latin-1', errors='ignore')

# extraction des contenus de <lem>
lemmas = re.findall(r'<LEM>(.*?)</LEM>', raw)
lemmas = [re.sub(r'\s+', ' ', l).strip() for l in lemmas if l and l.strip()]

# clés normalisées
dmf_lemma_keys = {norm_key(l) for l in lemmas}

print(f"Entrées <LEM> extraites: {len(lemmas)}")
print(f"Clés normalisées distinctes: {len(dmf_lemma_keys)}")

# aperçu
lemmas[:20]

## Construire la table d’alignement (corpus vers liste manuelle vers DMF)

Règles :
- on part des noms propres observés dans le TSV (fréquences) ;
- on joint la liste manuelle (si une correspondance base vers DMF est donnée, on la garde) ;
- on vérifie si le lemme (ou la forme DMF) existe dans l’inventaire DMF.

Sorties :
- `present_in_dmf` : le lemme du corpus existe tel quel dans DMF ;
- `mapped_to_dmf` : la liste ODS propose une forme DMF ;
- `mapped_dmf_exists` : cette forme DMF existe dans DMF ;
- `needs_modeling` : à modéliser/ajouter (pas trouvé + pas de mapping valide).

In [ ]:
# Jointure sur la clé normalisée
np_map = np_ods[['base_form','base_key','dmf_form','dmf_key']].drop_duplicates('base_key')

align = propn_counts.merge(np_map, left_on='lemma_key', right_on='base_key', how='left')

# Statuts
align['present_in_dmf'] = align['lemma_key'].isin(dmf_lemma_keys)
align['mapped_to_dmf'] = align['dmf_key'].notna() & (align['dmf_key'].astype(str).str.len()>0)
align['mapped_dmf_exists'] = align['dmf_key'].apply(lambda k: False if pd.isna(k) else (k in dmf_lemma_keys))

# Besoin de modélisation : ni présent tel quel, ni mapping valide vers un lemme DMF existant
align['needs_modeling'] = (~align['present_in_dmf']) & (~align['mapped_dmf_exists'])

# Ordonner par fréquence
align = align.sort_values('count', ascending=False).reset_index(drop=True)
align.head(60)

## Exports (diagnostics + TEI minimal)

On écrit dans `documentation/diagnostics/` :
- `_alignment.tsv` : table complète ;
- `_needs_modeling.tsv` : uniquement ceux à modéliser ;
- `_summary.json` : KPI ;
- `_to_model.tei.xml` : TEI minimal pour démarrer la modélisation.

### TEI minimal
Pour chaque nom propre à modéliser, on crée une entrée `<entry>` :
- lemme (forme canonique)
- POS = `NOMpro`
- un espace pour l’alignement DMF si cible connue

> à enrichir par la suite (type: personne/lieu, variantes, attestations, notes, etc.).

In [ ]:
out_dir = repo_root/'documentation'/'diagnostics'
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')

#path_all = out_dir/f'{stamp}_alignment.tsv'
#path_need = out_dir/f'{stamp}_needs_modeling.tsv'
#path_sum = out_dir/f'{stamp}_summary.json'
#path_tei = out_dir/f'{stamp}_to_model.tei.xml'

path_all = out_dir/f'{stamp}_alignment.tsv'
path_need = out_dir/f'{stamp}_needs_modeling.tsv'
path_sum = out_dir/f'{stamp}_summary.json'
path_tei = out_dir/f'{stamp}_to_model.tei.xml'

align.to_csv(path_all, sep='	', index=False)
align[align['needs_modeling']].to_csv(path_need, sep='	', index=False)

summary = {
    'tsv': str(TSV_PATH.relative_to(repo_root)),
    'propn_pos': sorted(PROPN_POS),
    'propn_token_count': int(len(propn)),
    'propn_type_count': int(len(propn_counts)),
    'present_in_dmf_types': int(align['present_in_dmf'].sum()),
    'mapped_dmf_exists_types': int(align['mapped_dmf_exists'].sum()),
    'needs_modeling_types': int(align['needs_modeling'].sum()),
}
with open(path_sum, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# TEI minimal

from xml.sax.saxutils import escape

def make_xml_id(s: str) -> str:
    # id stable à partir de la clé normalisée
    k = norm_key(s)
    k = re.sub(r'[^a-z0-9]+', '_', k).strip('_')
    return f'np_{k[:60]}' if k else 'np_unknown'

rows = align[align['needs_modeling']].copy()

tei_lines = []
tei_lines.append('<?xml version="1.0" encoding="UTF-8"?>')
tei_lines.append('<TEI xmlns="http://www.tei-c.org/ns/1.0">')
tei_lines.append('  <teiHeader>')
tei_lines.append('    <fileDesc>')
tei_lines.append('      <titleStmt><title>Noms propres à modéliser (OF3C vers DMF)</title></titleStmt>')
tei_lines.append('      <publicationStmt><p>Généré automatiquement à partir du corpus OF3C</p></publicationStmt>')
tei_lines.append('      <sourceDesc><p>Extrait du corpus OF3C et comparé au DMF corrigé du projet.</p></sourceDesc>')
tei_lines.append('    </fileDesc>')
tei_lines.append('  </teiHeader>')
tei_lines.append('  <text><body>')
tei_lines.append('    <list>')

for _,r in rows.iterrows():
    lemma = str(r['lemma'])
    xml_id = make_xml_id(lemma)
    tei_lines.append(f'      <entry xml:id="{xml_id}">')
    tei_lines.append('        <form type="lemma">')
    tei_lines.append(f'          <orth>{escape(lemma)}</orth>')
    tei_lines.append('        </form>')
    tei_lines.append('        <gramGrp><pos>NOMpro</pos></gramGrp>')
    tei_lines.append(f'        <note type="corpusFrequency">{int(r["count"])}</note>')
    # si la liste ODS proposait une forme DMF (même si absente), on la note
    if pd.notna(r.get('dmf_form')) and str(r.get('dmf_form')).strip():
        tei_lines.append(f'        <note type="suggestedDMF">{escape(str(r.get("dmf_form")).strip())}</note>')
    tei_lines.append('      </entry>')

tei_lines.append('    </list>')
tei_lines.append('  </body></text>')
tei_lines.append('</TEI>')

path_tei.write_text('\n'.join(tei_lines), encoding='utf-8')

path_all, path_need, path_sum, path_tei

In [ ]:
# je propose ici une alternative d'encodage conforme à la structure spécifique de l'XML source
# pour faciliter l'intégration dans le référentiel existant
out_dir = repo_root / 'documentation' / 'diagnostics'
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')

path_all = out_dir / f'{stamp}_alignment.tsv'
path_need = out_dir / f'{stamp}_needs_modeling.tsv'
path_sum = out_dir / f'{stamp}_summary.json'
path_xml = out_dir / f'{stamp}_bis_to_model.xml'

align.to_csv(path_all, sep='\t', index=False)
align[align['needs_modeling']].to_csv(path_need, sep='\t', index=False)

summary = {
    'tsv': str(TSV_PATH.relative_to(repo_root)),
    'propn_pos': sorted(PROPN_POS),
    'propn_token_count': int(len(propn)),
    'propn_type_count': int(len(propn_counts)),
    'present_in_dmf_types': int(align['present_in_dmf'].sum()),
    'mapped_dmf_exists_types': int(align['mapped_dmf_exists'].sum()),
    'needs_modeling_types': int(align['needs_modeling'].sum()),
}
with open(path_sum, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

from xml.sax.saxutils import escape
import pandas as pd

rows = align[align['needs_modeling']].copy()

xml_lines = []
xml_lines.append('<?xml version="1.0" encoding="UTF-8"?>')
xml_lines.append('<DMF version="2023">')

for _, r in rows.iterrows():
    lemma = str(r['lemma']).strip()

    xml_lines.append('    <ART>')
    xml_lines.append(f'        <VED>{escape(lemma)}</VED>')
    xml_lines.append('        <CODE>NOMpro</CODE>')
    xml_lines.append(f'        <LEM>{escape(lemma)}</LEM>')
    xml_lines.append('    </ART>')

xml_lines.append('</DMF>')

path_xml.write_text('\n'.join(xml_lines), encoding='utf-8')

path_all, path_need, path_sum, path_xml

## Lecture des résultats

- `alignment.tsv` : tout
- `needs_modeling.tsv` : ce qui n’est ni dans le DMF, ni mappé vers une forme DMF existante
- `to_model.tei.xml` : squelette TEI pour commencer